In [ ]:
from twilio.rest import Client
from datetime import datetime
import pandas as pd
import schedule
import time
import os

account_sid = "ur id"
auth_token = "ur token"

client = Client(account_sid, auth_token)

sent_indices = set()  # track which rows have already been sent

def send_whatsapp(phone: str, msg: str) -> None:
    message = client.messages.create(
        from_="whatsapp:+14155238886",
        body=msg,
        to=f"whatsapp:{phone}",
    )
    print(f"[{datetime.now()}] Sent to {phone}: {message.sid}")

def check_messages() -> None:
    df = pd.read_excel("scheduled_messages.xlsx")
    now = datetime.now()
    current_hhmm = (now.hour, now.minute)

    for index, row in df.iterrows():
        if index in sent_indices:
            continue

        send_time = row["SendTime"]

        # Handle both datetime/time objects and strings like "14:05"
        if isinstance(send_time, str):
            try:
                t = datetime.strptime(send_time.strip(), "%H:%M")
                row_hhmm = (t.hour, t.minute)
            except ValueError:
                print(f"Row {index}: unrecognized time format '{send_time}'")
                continue
        elif hasattr(send_time, "hour"):
            row_hhmm = (send_time.hour, send_time.minute)
        else:
            print(f"Row {index}: unexpected SendTime type {type(send_time)}")
            continue

        if row_hhmm == current_hhmm:
            send_whatsapp(str(row["Phone"]), str(row["Message"]))
            sent_indices.add(index)

schedule.every(1).minutes.do(check_messages)
print("Scheduler running...")

while True:
    schedule.run_pending()
    time.sleep(1)